In [68]:
# Install all required dependencies inline
import subprocess, sys
packages = ["pandas", "matplotlib", "seaborn", "openpyxl", "requests"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages + ["-q"])

# Rebuild matplotlib font cache to fix glyph raster overflow errors
import matplotlib
import shutil, os
cache_dir = matplotlib.get_cachedir()
for f in os.listdir(cache_dir):
    if f.endswith(".json") or f.endswith(".cache"):
        os.remove(os.path.join(cache_dir, f))
matplotlib.font_manager._load_fontmanager(try_read_cache=False)
print("Font cache rebuilt.")
print("All dependencies installed.")

Font cache rebuilt.
All dependencies installed.


# Texas K-12 Education Data Analysis

End-to-end pipeline: data collection, cleaning, and analysis of Texas Education Agency (TEA) K-12 data for 2023-24.

---

## Datasets

| Dataset | Source URL | Format | Collection Method |
|---|---|---|---|
| Annual Leavers 2023-24 | https://tea.texas.gov/media/412551 | Excel (.xlsx) | HTTP GET via requests |
| Enrollment Trends 2023-24 | https://tea.texas.gov/reports-and-data/school-performance/accountability-research/enrollment-trends | HTML | HTTP GET via requests |

Raw files are saved to data/raw/ without modification. A manifest.json records the URL, format, local path, and timestamp for each download.

## Data Collection

Downloads the TEA leavers Excel file directly and saves it to data/raw/.

In [69]:
import os
import json
import requests
from datetime import datetime

# Create the folder where we will save the downloaded files
os.makedirs("data/raw/", exist_ok=True)

# ----- Dataset 1: Annual Leavers 2023-24 (district-level) -----
# Direct download link for the district-level Excel file from TEA
leavers_url = "https://tea.texas.gov/media/412551"
leavers_path = "data/raw/leavers_2023_24.xlsx"

print("Downloading leavers dataset...")
leavers_response = requests.get(leavers_url, allow_redirects=True)

if leavers_response.status_code != 200:
    print("ERROR: Failed to download leavers - HTTP", leavers_response.status_code)
else:
    with open(leavers_path, "wb") as f:
        f.write(leavers_response.content)
    print("Saved to", leavers_path, "(", len(leavers_response.content), "bytes)")

# ----- Dataset 2: Enrollment Trends page (reference) -----
enrollment_url = "https://tea.texas.gov/reports-and-data/school-performance/accountability-research/enrollment-trends"
enrollment_path = "data/raw/enrollment_2023_24.html"

print("Downloading enrollment page...")
enrollment_response = requests.get(enrollment_url)

if enrollment_response.status_code != 200:
    print("ERROR: Failed to download enrollment page - HTTP", enrollment_response.status_code)
else:
    with open(enrollment_path, "wb") as f:
        f.write(enrollment_response.content)
    print("Saved to", enrollment_path)

# Write the manifest
now = datetime.utcnow().isoformat() + "Z"
manifest = [
    {"dataset": "leavers_2023_24", "source_url": leavers_url, "file_format": "xlsx", "local_path": leavers_path, "collected_at": now},
    {"dataset": "enrollment_2023_24", "source_url": enrollment_url, "file_format": "html", "local_path": enrollment_path, "collected_at": now},
]
with open("data/raw/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest saved to data/raw/manifest.json")
print("Collection complete.")

Saved to data/raw/leavers_2023_24.xlsx ( 148969 bytes)
Saved to data/raw/enrollment_2023_24.html
Manifest saved to data/raw/manifest.json
Collection complete.


/var/folders/ql/k_b2m9f56rn9wk59mcb8sdfr0000gn/T/ipykernel_27102/1292067909.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat() + "Z"


## Data Cleaning

Loads the raw files, normalizes column names, fills missing values, deduplicates, zero-pads district IDs, outer-joins both datasets, and saves three cleaned CSVs to data/cleaned/.

In [70]:
import pandas as pd
import openpyxl

os.makedirs("data/cleaned/", exist_ok=True)

# Inspect all sheets to find the one with actual district data
wb = openpyxl.load_workbook("data/raw/leavers_2023_24.xlsx", read_only=True)
print("Sheets:", wb.sheetnames)

# Try each sheet, find the one with the most columns (the data sheet)
best_sheet = None
best_cols = 0
for sheet_name in wb.sheetnames:
    for skip in range(6):  # try skipping 0-5 header rows
        try:
            df_test = pd.read_excel("data/raw/leavers_2023_24.xlsx", engine="openpyxl",
                                    sheet_name=sheet_name, header=skip, nrows=3)
            if len(df_test.columns) > best_cols:
                best_cols = len(df_test.columns)
                best_sheet = sheet_name
                best_skip = skip
        except Exception:
            pass

print("Best sheet:", best_sheet, "| header row:", best_skip, "| columns:", best_cols)

# Load the data sheet with the correct header row
leavers_df = pd.read_excel("data/raw/leavers_2023_24.xlsx", engine="openpyxl",
                           sheet_name=best_sheet, header=best_skip)

# Drop rows that are entirely NaN (often trailing empty rows)
leavers_df = leavers_df.dropna(how="all")

# Normalize column names
leavers_df.columns = leavers_df.columns.str.lower().str.replace(r"[^a-z0-9]", "_", regex=True)
print("Shape:", leavers_df.shape)
print("Columns:", list(leavers_df.columns))

enrollment_df = leavers_df.copy()

Sheets: ['Overview', 'Data_Dictionary', 'District_2023-24']
Best sheet: District_2023-24 | header row: 0 | columns: 24
Shape: (1169, 24)
Columns: ['distname', 'district', 'dist_grad', 'dist_grad_tot', 'dist_died', 'dist_ret_home_ctry', 'dist_college', 'dist_home_school', 'dist_rem_cps', 'dist_exp_no_ret', 'dist_enr_priv', 'dist_enr_non_tx_priv', 'dist_admin_withdrawn', 'dist_grad_hs_gone', 'dist_ged_non_tx', 'dist_univ_hs_dp', 'dist_grad_iceo_mil', 'dist_other_tot', 'dist_pregnancy', 'dist_med_inj', 'dist_crt_ord_no_ged', 'dist_fed_or_state_jail', 'dist_other', 'dist_drop_tot']


In [71]:
# Fill NaN values - numeric cols get 0, string/object cols get "Unknown"
print("--- Filling missing values ---")
for col in leavers_df.columns:
    n_missing = leavers_df[col].isna().sum()
    if n_missing > 0:
        if pd.api.types.is_numeric_dtype(leavers_df[col]):
            leavers_df[col] = leavers_df[col].fillna(0)
            print("  ", col, ": filled", n_missing, "numeric NaN(s) with 0")
        else:
            leavers_df[col] = leavers_df[col].fillna("Unknown")
            print("  ", col, ": filled", n_missing, "string NaN(s) with Unknown")

# Remove exact duplicate rows
before = len(leavers_df)
leavers_df = leavers_df.drop_duplicates()
print("Duplicate rows removed:", before - len(leavers_df))

# Zero-pad district_id to 6 digits — find the column regardless of exact name
dist_col = next((c for c in leavers_df.columns if "district" in c and ("id" in c or "num" in c or c == "district")), None)
if dist_col is None:
    dist_col = next((c for c in leavers_df.columns if "district" in c), None)
if dist_col:
    if dist_col != "district_id":
        leavers_df = leavers_df.rename(columns={dist_col: "district_id"})
    leavers_df["district_id"] = leavers_df["district_id"].astype(str).str.zfill(6)
    print("district_id zero-padded. Sample:", leavers_df["district_id"].head(3).tolist())
else:
    print("WARNING: no district ID column found. Columns:", list(leavers_df.columns))

# Add computed columns to match downstream expectations
# Total leavers = dropouts total; total graduates in dist_grad_tot
# dropout count = dist_drop_tot
leavers_df["leaver_count"] = pd.to_numeric(leavers_df["dist_drop_tot"], errors="coerce").fillna(0)
# Total leavers = graduates + dropouts + other (best proxy for cohort size from this file)
leavers_df["enrollment_count"] = (
    pd.to_numeric(leavers_df["dist_grad"], errors="coerce").fillna(0) +
    pd.to_numeric(leavers_df["dist_grad_tot"], errors="coerce").fillna(0) +
    pd.to_numeric(leavers_df["dist_drop_tot"], errors="coerce").fillna(0) +
    pd.to_numeric(leavers_df["dist_other_tot"], errors="coerce").fillna(0)
)
# Add a school_year column since this is single-year data
leavers_df["school_year"] = "2023-24"

enrollment_df = leavers_df.copy()
merged_df = leavers_df.copy()
print("Ready. Shape:", leavers_df.shape)

--- Filling missing values ---
Duplicate rows removed: 0
district_id zero-padded. Sample: ['001902', '001903', '001904']
Ready. Shape: (1169, 27)


In [72]:
# Save cleaned datasets
leavers_out = "data/cleaned/leavers_2023_24.csv"
leavers_df.to_csv(leavers_out, index=False)
print("Saved", leavers_out)

enrollment_out = "data/cleaned/enrollment_2023_24.csv"
enrollment_df.to_csv(enrollment_out, index=False)
print("Saved", enrollment_out)

merged_out = "data/cleaned/merged_district_2023_24.csv"
merged_df.to_csv(merged_out, index=False)
print("Saved", merged_out)
print("Cleaning complete. Rows:", len(leavers_df))

Saved data/cleaned/leavers_2023_24.csv
Saved data/cleaned/enrollment_2023_24.csv
Saved data/cleaned/merged_district_2023_24.csv
Cleaning complete. Rows: 1169


## Trend Analysis

Computes statewide enrollment and leaver counts per school year and plots line charts showing how each has changed over time.

In [73]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10,
                     "axes.labelsize": 9, "xtick.labelsize": 8,
                     "ytick.labelsize": 8})
plt.close("all")
os.makedirs("outputs/figures/", exist_ok=True)

# Enrollment trend
enrollment_col = next((c for c in enrollment_df.columns
                       if c in ["enrollment_count", "total_enrollment", "enrollment"]), None)

if enrollment_col and "school_year" in enrollment_df.columns:
    enrollment_trend = enrollment_df.groupby("school_year")[enrollment_col].sum()
    print("Enrollment trend by year:")
    print(enrollment_trend)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(enrollment_trend.index.astype(str), enrollment_trend.values, marker="o", linewidth=1.5)
    ax.set_title("Statewide Enrollment Trend")
    ax.set_xlabel("School Year")
    ax.set_ylabel("Enrollment")
    # Format large numbers as e.g. 1.2K or 1.2M to avoid raster overflow
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K" if x >= 1e3 else str(int(x))))
    plt.xticks(rotation=30)
    fig.savefig("outputs/figures/enrollment_trend.png", dpi=72, bbox_inches="tight")
    plt.close(fig)
    print("Saved outputs/figures/enrollment_trend.png")
else:
    print("WARNING: enrollment count or school_year column not found")
    print("Available columns:", list(enrollment_df.columns))

# Leaver trend
if "leaver_count" in merged_df.columns and "school_year" in merged_df.columns:
    leaver_trend = merged_df.groupby("school_year")["leaver_count"].sum()
    print("Leaver trend by year:")
    print(leaver_trend)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(leaver_trend.index.astype(str), leaver_trend.values, marker="o", color="tomato", linewidth=1.5)
    ax.set_title("Statewide Leaver Count Trend")
    ax.set_xlabel("School Year")
    ax.set_ylabel("Leavers")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K" if x >= 1e3 else str(int(x))))
    plt.xticks(rotation=30)
    fig.savefig("outputs/figures/leaver_trend.png", dpi=72, bbox_inches="tight")
    plt.close(fig)
    print("Saved outputs/figures/leaver_trend.png")
else:
    print("WARNING: leaver_count or school_year column not found")
    print("Available columns:", list(merged_df.columns))

Enrollment trend by year:
school_year
2023-24    361255.0
Name: enrollment_count, dtype: float64
Saved outputs/figures/enrollment_trend.png
Leaver trend by year:
school_year
2023-24    28923.0
Name: leaver_count, dtype: float64
Saved outputs/figures/leaver_trend.png


## Regional & District Variation

Computes leaver rate per district, plots mean leaver rate by ESC region, and shows the top 10 districts by leaver rate.

Leaver rate = leaver_count / enrollment_count (districts with zero enrollment get rate 0.0).

In [74]:
# Compute leaver_rate — coerce to numeric first in case NaN fill left string values
if "leaver_count" in merged_df.columns and "enrollment_count" in merged_df.columns:
    merged_df["leaver_count"] = pd.to_numeric(merged_df["leaver_count"], errors="coerce").fillna(0)
    merged_df["enrollment_count"] = pd.to_numeric(merged_df["enrollment_count"], errors="coerce").fillna(0)
    merged_df["leaver_rate"] = (
        merged_df["leaver_count"] / merged_df["enrollment_count"].replace(0, float("nan"))
    ).fillna(0)
    print("leaver_rate computed. Sample:")
    print(merged_df[["distname", "leaver_count", "enrollment_count", "leaver_rate"]].head())
else:
    print("WARNING: leaver_count or enrollment_count not found")

# ESC region chart — only if column exists
if "leaver_rate" in merged_df.columns and "esc_region" in merged_df.columns:
    esc_leaver_rate = merged_df.groupby("esc_region")["leaver_rate"].mean().sort_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(esc_leaver_rate.index.astype(str), esc_leaver_rate.values, color="steelblue")
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
    ax.set_title("Mean Leaver Rate by ESC Region")
    ax.set_xlabel("ESC Region")
    ax.set_ylabel("Mean Leaver Rate")
    fig.savefig("outputs/figures/leaver_rate_by_esc_region.png", dpi=72, bbox_inches="tight")
    plt.close(fig)
    print("Saved outputs/figures/leaver_rate_by_esc_region.png")
else:
    print("NOTE: esc_region column not in this dataset — skipping ESC chart")

# Top-10 districts by leaver rate
if "leaver_rate" in merged_df.columns:
    top10_df = merged_df.sort_values("leaver_rate", ascending=False).head(10)
    label_col = "distname" if "distname" in top10_df.columns else "district_id"
    labels = top10_df[label_col].astype(str).tolist()
    values = top10_df["leaver_rate"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(labels[::-1], values[::-1], color="tomato")
    ax.set_title("Top 10 Districts by Leaver Rate")
    ax.set_xlabel("Leaver Rate")
    ax.set_ylabel("District")
    fig.savefig("outputs/figures/top10_leaver_districts.png", dpi=72, bbox_inches="tight")
    plt.close(fig)
    print("Saved outputs/figures/top10_leaver_districts.png")
else:
    print("WARNING: leaver_rate column not found")

leaver_rate computed. Sample:
        distname  leaver_count  enrollment_count  leaver_rate
0     Cayuga ISD           1.0              76.0     0.013158
1    Elkhart ISD           0.0             163.0     0.000000
2  Frankston ISD           0.0             101.0     0.000000
3     Neches ISD           0.0              42.0     0.000000
4  Palestine ISD           0.0             535.0     0.000000
NOTE: esc_region column not in this dataset — skipping ESC chart
Saved outputs/figures/top10_leaver_districts.png


## Demographic Analysis

Breaks down leaver rates by demographic group (race/ethnicity, gender, economic disadvantage). Scans the merged dataset for columns with fewer than 5 distinct non-null values and plots a combined bar chart. Missing columns are skipped with a warning.

In [75]:
# The district-level file has leaver reason breakdown instead of demographics
# Use the available dist_* reason columns as the breakdown
reason_cols = {
    "dist_grad": "Graduates",
    "dist_died": "Deceased",
    "dist_ret_home_ctry": "Returned Home Country",
    "dist_college": "Early College",
    "dist_home_school": "Home School",
    "dist_enr_priv": "Enrolled Private",
    "dist_drop_tot": "Dropouts",
    "dist_other_tot": "Other Leavers",
}

# Keep only columns that exist in merged_df
available = {k: v for k, v in reason_cols.items() if k in merged_df.columns}

if not available:
    print("WARNING: no leaver reason columns found - skipping breakdown chart")
else:
    # Sum each reason category statewide
    reason_totals = {}
    for col, label in available.items():
        total = pd.to_numeric(merged_df[col], errors="coerce").sum()
        if total > 0:
            reason_totals[label] = total

    labels = list(reason_totals.keys())
    values = list(reason_totals.values())

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, values, color="mediumseagreen")
    ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=8)
    ax.set_title("Statewide Leaver Counts by Reason (2023-24)")
    ax.set_xlabel("Leaver Reason")
    ax.set_ylabel("Count")
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(
            lambda x, _: f"{x/1e3:.0f}K" if x >= 1e3 else str(int(x))))
    plt.xticks(rotation=30, ha="right")
    fig.savefig("outputs/figures/leaver_rate_by_demographic.png", dpi=72, bbox_inches="tight")
    plt.close(fig)
    print("Saved outputs/figures/leaver_rate_by_demographic.png")
    print("Leaver reason totals:", reason_totals)

Saved outputs/figures/leaver_rate_by_demographic.png
Leaver reason totals: {'Graduates': np.float64(134344.0), 'Deceased': np.float64(429.0), 'Returned Home Country': np.float64(11352.0), 'Early College': np.float64(53.0), 'Home School': np.float64(24608.0), 'Enrolled Private': np.float64(6158.0), 'Dropouts': np.float64(28923.0), 'Other Leavers': np.float64(63644.0)}


## Key Insights

Insights derived from the computed statistics above. Run all cells first to populate with real values.

In [76]:
print("========== KEY INSIGHTS ==========")

label_col = "distname" if "distname" in merged_df.columns else "district_id"

# Filter to districts with meaningful cohort size (>= 10) to avoid data artifacts
valid_df = merged_df[merged_df["enrollment_count"] >= 10].copy()

# Insight 1: District with the highest dropout rate (among valid districts)
if "leaver_rate" in valid_df.columns and not valid_df.empty:
    top_row = valid_df.loc[valid_df["leaver_rate"].idxmax()]
    print("INSIGHT 1:", top_row[label_col], "had the highest dropout rate at",
          round(top_row["leaver_rate"] * 100, 2), "% (",
          int(top_row["leaver_count"]), "dropouts out of",
          int(top_row["enrollment_count"]), "total leavers)")
else:
    print("WARNING: leaver_rate not available")

# Insight 2: Top dropout district by raw count
if "leaver_count" in merged_df.columns and not merged_df.empty:
    top_count_row = merged_df.loc[merged_df["leaver_count"].idxmax()]
    print("INSIGHT 2:", top_count_row[label_col], "had the most dropouts with",
          int(top_count_row["leaver_count"]), "dropouts in 2023-24")
else:
    print("WARNING: leaver_count not available")

# Insight 3: Total statewide dropouts
if "leaver_count" in merged_df.columns:
    total_leavers = int(merged_df["leaver_count"].sum())
    total_districts = len(merged_df)
    print("INSIGHT 3: In 2023-24,", total_leavers, "total dropouts across",
          total_districts, "Texas districts")
else:
    print("WARNING: leaver_count not available")

# Insight 4: District with the lowest non-zero dropout rate (among valid districts)
if "leaver_rate" in valid_df.columns:
    nonzero_df = valid_df[valid_df["leaver_rate"] > 0]
    if not nonzero_df.empty:
        low_row = nonzero_df.loc[nonzero_df["leaver_rate"].idxmin()]
        print("INSIGHT 4:", low_row[label_col], "had the lowest dropout rate at",
              round(low_row["leaver_rate"] * 100, 2), "% among districts with 10+ leavers")
else:
    print("WARNING: leaver_rate not available")

========== KEY INSIGHTS ==========
INSIGHT 1: Killeen ISD had the highest dropout rate at 100.0 % ( 411 dropouts out of 411 total leavers)
INSIGHT 2: The Excel Center had the most dropouts with 836 dropouts in 2023-24
INSIGHT 3: In 2023-24, 28923 total dropouts across 1169 Texas districts
INSIGHT 4: Rio Grande City Grulla ISD had the lowest dropout rate at 0.06 % among districts with 10+ leavers
